In [ ]:
!pip install openai

In [ ]:
import os
from google.colab import userdata

In [ ]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
from openai import OpenAI
import json

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
model_name="gemini-1.5-flash"

In [ ]:
def get_chatbot_response(client,model_name,messages,temperature=0):
    input_messages = []
    for message in messages:
        input_messages.append({"role": message["role"], "content": message["content"]})

    response = client.chat.completions.create(
        model=model_name,
        messages=input_messages,
        temperature=temperature,
        top_p=0.8,
        max_tokens=2000,
    ).choices[0].message.content

    return response

In [ ]:
messages = [{'role':'user','content':"What's the Agentic AI?"}]
response = get_chatbot_response(client,model_name,messages)
print(response)

There's no single, universally defined "Agentic AI."  The term describes a type of artificial intelligence that exhibits agency – the capacity to act independently and pursue its own goals.  It's a concept rather than a specific technology.  Different researchers and developers have varying interpretations of what constitutes an agentic AI, but some key characteristics generally include:

* **Goal-directed behavior:** Agentic AI systems don't simply react to inputs; they actively pursue goals and objectives.  These goals might be explicitly programmed, learned through reinforcement learning, or even emergent from the system's internal dynamics.

* **Proactive planning and decision-making:**  Instead of passively waiting for instructions, agentic AI systems can plan sequences of actions to achieve their goals, considering potential obstacles and adapting to changing circumstances.

* **Autonomous operation:**  They operate with a degree of independence, requiring minimal human intervent

In [ ]:
def get_chatbot_response(client, model_name, messages, temperature=0):
    input_messages = []
    for message in messages:
        input_messages.append({"role": message["role"], "content": message["content"]})

    response = client.chat.completions.create(
        model=model_name,
        messages=input_messages,
        temperature=temperature,
        top_p=0.8,
        max_tokens=2000,
    ).choices[0].message.content

    # Check if response is a valid JSON string
    try:
        json.loads(response)  # Attempt to parse as JSON
    except json.JSONDecodeError:
        # If it fails, wrap it in a valid JSON format
        response = json.dumps({"message": response})

    return response

In [ ]:
from typing import Protocol, List, Dict, Any

class AgentProtocol(Protocol):
    def get_response(self, messages: List[Dict[str, Any]]) -> Dict[str, Any]:
        ...

In [ ]:
# best guard agent
import json
import os
from copy import deepcopy
from openai import OpenAI

class GuardAgent:
    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("GOOGLE_API_KEY"),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
        )
        self.model_name = "gemini-1.5-flash"

    def get_response(self, messages):
        messages = deepcopy(messages)

        # system_prompt = """
        # You are a helpful AI assistant for a coffee shop application which serves drinks and pastries.
        # ...
        # """
        system_prompt = """
            You are a helpful AI assistant for a coffee shop application which serves drinks and pastries. Your task is to determine whether the user is asking something relevant to the coffee shop or not.

            The user is allowed to:
            1. Ask questions about the coffee shop, like location, working hours, menu items, and coffee shop-related questions.
            2. Ask questions about menu items, including ingredients and more details about the items.
            3. Make an order.
            4. Ask about recommendations of what to buy.

            The user is NOT allowed to:
            1. Ask questions about anything else other than our coffee shop.
            2. Ask questions about the staff or how to make a certain menu item.

            Your output should be in a structured JSON format as follows. Each key is a string, and each value is a string. Make sure to follow the format exactly:
            {
                "chain of thought": "Go over each of the points above and determine if the message falls under these points. Then write your thoughts about which point this input is relevant to.",
                "decision": "allowed" or "not allowed". Pick one of those, and only write the word,
                "message": Leave the message empty "" if it's allowed; otherwise, write "Sorry, I can't help with that. Can I help you with your order?"
            }

            """

        input_messages = [{"role": "system", "content": system_prompt}] + messages[-3:]
        chatbot_output = get_chatbot_response(self.client, self.model_name, input_messages)
        output = self.postprocess(chatbot_output)

        return output

        # # Print the raw output for debugging
        # print("Raw chatbot output:", chatbot_output)

        # return self.postprocess(chatbot_output)

    # def postprocess(self,output):
    #     output = json.loads(output)

    #     dict_output = {
    #         "role": "assistant",
    #         "content": output['message'],
    #         "memory": {"agent":"guard_agent",
    #                 "guard_decision": output['decision']
    #                 }
    #     }
    #     return dict_output

    def postprocess(self, output):
        parsed_output = json.loads(output)
        dict_output = {
            "role": "assistant",
            "content": parsed_output.get('message', ''),
            "memory": {
                "agent": "guard_agent",
                "guard_decision": parsed_output.get('decision', '')
            }
        }
        return dict_output


    # def postprocess(self, output):
    #     try:
    #         parsed_output = json.loads(output)
    #         dict_output = {
    #             "role": "assistant",
    #             "content": parsed_output.get('message', ''),
    #             "memory": {
    #                 "agent": "guard_agent",
    #                 "guard_decision": parsed_output.get('decision', '')
    #             }
    #         }
    #     except json.JSONDecodeError as e:
    #         dict_output = {
    #             "role": "assistant",
    #             "content": "Error parsing the response.",
    #             "memory": {
    #                 "agent": "guard_agent",
    #                 "guard_decision": "not allowed"
    #             }
    #         }
    #         print(f"Error decoding JSON: {e}")

    #     return dict_output




In [ ]:
if __name__ == "__main__":
    guard_agent = GuardAgent()
    # classification_agent = ClassificationAgent()

    messages = []

    while True:
        print("\n\nPrint messages.......")
        for message in messages:
            print(f"{message['role']}: {message['content']}")

        prompt = input("User: ")
        if prompt.lower() == "exit":
            print("Exiting the conversation.")
            break

        messages.append({"role": "user", "content": prompt})


        # Guard agent response
        guard_agent_response = guard_agent.get_response(messages)
        print("Guard agent output: ", guard_agent_response)#for checking purpose

        # if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        #     messages.append(guard_agent_response)
        #     continue
        messages.append(guard_agent_response)#for checking single guard agent



Print messages.......
User: what's 1+3
Guard agent output:  {'role': 'assistant', 'content': '```json\n{\n  "chain of thought": "The user is asking a question about a simple math problem, which is not related to the coffee shop, its menu, or making an order.  Therefore, it\'s not allowed.",\n  "decision": "not allowed",\n  "message": "Sorry, I can\'t help with that. Can I help you with your order?"\n}\n```\n', 'memory': {'agent': 'guard_agent', 'guard_decision': ''}}


Print messages.......
user: what's 1+3
assistant: ```json
{
  "chain of thought": "The user is asking a question about a simple math problem, which is not related to the coffee shop, its menu, or making an order.  Therefore, it's not allowed.",
  "decision": "not allowed",
  "message": "Sorry, I can't help with that. Can I help you with your order?"
}
```

User: exit
Exiting the conversation.


In [ ]:
import os
import json
from copy import deepcopy

from openai import OpenAI


class ClassificationAgent():
    def __init__(self):

        self.client = OpenAI(

            api_key=os.getenv("GOOGLE_API_KEY"),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
            )
        self.model_name="gemini-1.5-flash"


    def get_response(self,messages):
        messages = deepcopy(messages)


        # system_prompt = """
        # You are a helpful AI assistant for a coffee shop application.
        # ...
        # """

        system_prompt = """
            You are a helpful AI assistant for a coffee shop application.
            Your task is to determine what agent should handle the user input. You have 3 agents to choose from:
            1. details_agent: This agent is responsible for answering questions about the coffee shop, like location, delivery places, working hours, details about menu items, or listing items in the menu.
            2. order_taking_agent: This agent is responsible for taking orders from the user. It’s responsible for having a conversation with the user about the order until it’s complete. The user may say phrases like “I want to order,” “I would like a latte,” or “Add a latte to my order.”
            3. recommendation_agent: This agent is responsible for giving recommendations to the user about what to buy. If the user asks for a recommendation, this agent should be used, such as “What should I buy?” or “Can you recommend something for me?”

            Your output should be in a structured json format like so. each key is a string and each value is a string. Make sure to follow the format exactly:
            {
            "chain of thought": "go over each of the agents above and write some of your thoughts about what agent is this input relevant to.",
            "decision": "details_agent" or "order_taking_agent" or "recommendation_agent". Pick one of those and only write the word.",
            "message": leave the message empty.
            }

            """

        input_messages = [
            {"role": "system", "content": system_prompt},
        ]

        input_messages += messages[-3:]

        chatbot_output =get_chatbot_response(self.client,self.model_name,input_messages)
        output = self.postprocess(chatbot_output)
        return output




    def postprocess(self, output):
        try:
            # Parse the output if it is a valid JSON
            output = json.loads(output)
        except json.JSONDecodeError:
            output = {"message": "Error parsing the response.", "decision": "details_agent"}  # Default to 'details_agent' in case of error.

        # Extract the message
        message = output.get('message', "").strip()
        decision = output.get('decision', "details_agent")  # Default to 'details_agent' if no decision is made.

        # Log the received message for debugging
        # print(f"User message: {message}")#for testing purpose this one need

        # Handle ordering intent more specifically
        order_keywords = ["order", "buy", "latte", "coffee", "want to", "get a", "add"]
        if any(keyword in message.lower() for keyword in order_keywords):
            if "latte" in message.lower() or "coffee" in message.lower():
                decision = "order_taking_agent"
            else:
                decision = "details_agent"  # If the message contains buying or ordering without specific drink mention, default to details.

        # Handle recommendation intent
        recommendation_keywords = ["recommend", "suggest", "what should", "what can", "what to"]
        if any(keyword in message.lower() for keyword in recommendation_keywords):
            decision = "recommendation_agent"

        # Handle informational queries (details agent)
        details_keywords = ["price", "menu", "hours", "location", "open", "available", "drink options"]
        if any(keyword in message.lower() for keyword in details_keywords):
            decision = "details_agent"

        # Log the decision for debugging
        # print(f"Decision made: {decision}")#for testing purpose this one need



        # Return the output in the correct format
        dict_output = {
            "role": "assistant",
            "content": message,
            "memory": {
                "agent": "classification_agent",
                "classification_decision": decision
            }
        }

        return dict_output

In [ ]:
if __name__ == "__main__":
    guard_agent = GuardAgent()
    classification_agent = ClassificationAgent()

    messages = []

    while True:
        print("\n\nPrint messages.......")
        for message in messages:
            print(f"{message['role']}: {message['content']}")

        prompt = input("User: ")
        if prompt.lower() == "exit":
            print("Exiting the conversation.")
            break

        messages.append({"role": "user", "content": prompt})


        # Guard agent response
        guard_agent_response = guard_agent.get_response(messages)
        # print("Guard agent output: ", guard_agent_response)

        if guard_agent_response["memory"]["guard_decision"] == "not allowed":
            messages.append(guard_agent_response)
            continue
        # messages.append(guard_agent_response)

        # Classification agent response
        classification_agent_response = classification_agent.get_response(messages)
        chosen_agent = classification_agent_response ["memory"]["classification_decision"]
        print("Chosen Agent: ",chosen_agent)



Print messages.......
User: one latte please
User message: ```json
{
  "chain of thought": "The user is clearly placing an order.  The phrase \"one latte please\" is a direct order request. The details agent is for informational queries, and the recommendation agent is for suggestions. Therefore, the order_taking_agent is the most appropriate.",
  "decision": "order_taking_agent",
  "message": ""
}
```
Decision made: recommendation_agent
Chosen Agent:  recommendation_agent


Print messages.......
user: one latte please
User: what recommend me
User message: ```json
{
  "chain of thought": "The user is asking for a recommendation on what to buy.  This falls directly under the purview of the recommendation agent.",
  "decision": "recommendation_agent",
  "message": ""
}
```
Decision made: recommendation_agent
Chosen Agent:  recommendation_agent


Print messages.......
user: one latte please
user: what recommend me
User: exit
Exiting the conversation.


In [ ]:
!pip install Pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 15.7 MB/s eta 0:00:00


In [ ]:
def get_embedding(embedding_client,model,text_input):
    output = embedding_client.embeddings.create(input = text_input,model=model)

    embedings = []
    for embedding_object in output.data:
        embedings.append(embedding_object.embedding)

    return embedings

In [ ]:
import os

In [ ]:
# Set the correct environment variable name
os.environ["PINECONE_API_KEY"] = "" # put pinecone api key here
os.environ["PINECONE_INDEX_NAME"] = "coffeeshop5"

In [ ]:
import os

from openai import OpenAI
from copy import deepcopy
from pinecone import Pinecone


class DetailsAgent():
    def __init__(self):
        self.client = OpenAI(

            api_key=os.getenv("GOOGLE_API_KEY"),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
        )
        self.model_name="gemini-1.5-flash"

        self.embedding_client = OpenAI(
            api_key=os.getenv("GOOGLE_API_KEY"),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
        )
        self.model="text-embedding-004"
        pinecone_api_key = os.getenv("PINECONE_API_KEY")
        self.index_name = os.getenv("PINECONE_INDEX_NAME")

        # Check if index_name is None
        if not self.index_name:
            raise ValueError("PINECONE_INDEX_NAME environment variable is not set.")

        self.pc = Pinecone(api_key=pinecone_api_key)


    def get_closest_results(self,index_name,input_embeddings,top_k=2):
        index = self.pc.Index(index_name)

        results = index.query(
            namespace="ns1",
            vector=input_embeddings,
            top_k=top_k,
            include_values=False,
            include_metadata=True
        )

        return results

    def get_response(self, messages):
        messages = deepcopy(messages)

        user_message = messages[-1]['content']
        embedding = get_embedding(self.embedding_client, self.model, user_message)[0]
        result = self.get_closest_results(self.index_name, embedding)
        source_knowledge = "\n".join([x['metadata']['text'].strip() + '\n' for x in result['matches']])

        prompt = f"""
        Using the contexts below, answer the query.

        Contexts:
        {source_knowledge}

        Query: {user_message}
        """

        system_prompt = """You are a customer support agent for a coffee shop called Merry's Way. You should answer every question as if you are a waiter and provide the necessary information to the user regarding their orders."""
        messages[-1]['content'] = prompt
        input_messages = [{"role": "system", "content": system_prompt}] + messages[-3:]

        chatbot_output = get_chatbot_response(self.client, self.model_name, input_messages)
        output = self.postprocess(chatbot_output)
        return output

    def postprocess(self,output):
        output = {
            "role": "assistant",
            "content": output,
            "memory": {"agent":"details_agent"
                      }
        }
        return output





In [ ]:
# this work beter and best
from typing import Dict
if __name__ == "__main__":
    # Instantiate your agents
    guard_agent = GuardAgent()
    classification_agent = ClassificationAgent()
    # details_agent = DetailsAgent()

    # Map of agent names to their instances
    agent_dict: Dict[str, AgentProtocol] = {
        "details_agent": DetailsAgent()
    }

    messages = []

    while True:
        print("\n\nPrint messages.......")
        for message in messages:
            print(f"{message['role']}: {message['content']}")

        prompt = input("User: ")
        if prompt.lower() == "exit":
            print("Exiting the conversation.")
            break

        messages.append({"role": "user", "content": prompt})

        # Guard agent response
        guard_agent_response = guard_agent.get_response(messages)

        if guard_agent_response["memory"]["guard_decision"] == "not allowed":
            messages.append(guard_agent_response)
            continue

        # Classification agent response
        classification_agent_response = classification_agent.get_response(messages)
        chosen_agent = classification_agent_response["memory"]["classification_decision"]
        print("Chosen Agent: ", chosen_agent)

        # # Get the chosen agent response
        # agent = agent_dict[chosen_agent]
        # response = agent.get_response(messages)

        # messages.append(response)
        # Get the chosen agent response
        agent = agent_dict.get(chosen_agent)
        if agent is None:
            print(f"Error: Agent '{chosen_agent}' not found.")
            continue

        response = agent.get_response(messages)
        messages.append(response)



Print messages.......
User: exit
Exiting the conversation.


In [ ]:
def double_check_json_output(client,model_name,json_string):
    prompt = f"""
    You will check this JSON string and correct any mistakes that will make it invalid. Then you will return the corrected JSON string. Nothing else.
    If the JSON is correct, just return it.

    If there is any text before the order after the JSON string, remove it.
    Do NOT return a single letter outside of the JSON string.
    Make sure that each key is enclosed in double quotes.
    The first thing you should write is the opening curly brace of the JSON, and the last character you write should be the closing curly brace.

    You should check the JSON string for the following text between triple backticks:

    {json_string}
    """

    messages = [{"role": "user", "content": prompt}]

    response = get_chatbot_response(client,model_name,messages)
    # response = response.replace("`","")

    return response

In [ ]:
# individual check all the function
import json
import pandas as pd
import os

from openai import OpenAI
from copy import deepcopy




class RecommendationAgent():
    def __init__(self,apriori_recommendation_path,popular_recommendation_path):
        self.client = OpenAI(

            api_key=os.getenv("GOOGLE_API_KEY"),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
        )
        self.model_name="gemini-1.5-flash"


        with open(apriori_recommendation_path, 'r') as file:
            self.apriori_recommendations = json.load(file)

        self.popular_recommendations = pd.read_csv(popular_recommendation_path)
        self.products = self.popular_recommendations['product'].tolist()
        self.product_categories = list(set(self.popular_recommendations['product_category'].tolist()))
    def get_apriori_recommendation(self,products,top_k=5):
        recommendation_list = []
        for product in products:
            if product in self.apriori_recommendations:
                recommendation_list += self.apriori_recommendations[product]

        # Sort recommendation list by "confidence"
        recommendation_list = sorted(recommendation_list,key=lambda x: x['confidence'],reverse=True)

        recommendations = []
        recommendations_per_category = {}
        for recommendation in recommendation_list:
            # If Duplicated recommendations then skip
            if recommendation in recommendations:
                continue

            # Limit 2 recommendations per category
            product_catory = recommendation['product_category']
            if product_catory not in recommendations_per_category:
                recommendations_per_category[product_catory] = 0

            if recommendations_per_category[product_catory] >= 2:
                continue

            recommendations_per_category[product_catory]+=1

            # Add recommendation
            recommendations.append(recommendation['product'])

            if len(recommendations) >= top_k:
                break

        return recommendations
    def get_popular_recommendation(self,product_categories=None,top_k=5):
        recommendations_df = self.popular_recommendations

        if type(product_categories) == str:
            product_categories = [product_categories]

        if product_categories is not None:
            recommendations_df = self.popular_recommendations[self.popular_recommendations['product_category'].isin(product_categories)]
        recommendations_df = recommendations_df.sort_values(by='number_of_transactions',ascending=False)

        if recommendations_df.shape[0] == 0:
            return []

        recommendations = recommendations_df['product'].tolist()[:top_k]
        return recommendations


if __name__ == "__main__":
     recommend_agent = RecommendationAgent("/content/apriori_recommendations.json","/content/popularity_recommendation.csv")

     print(recommend_agent.get_apriori_recommendation(['Latte']))
     print(recommend_agent.get_popular_recommendation(product_categories='Bakery'))
     print(recommend_agent.get_popular_recommendation())


['Sugar Free Vanilla syrup', 'Carmel syrup', 'Croissant', 'Chocolate Croissant']
['Chocolate Croissant', 'Ginger Scone', 'Jumbo Savory Scone', 'Croissant', 'Chocolate Chip Biscotti']
['Cappuccino', 'Latte', 'Dark chocolate', 'Chocolate Croissant', 'Espresso shot']


In [ ]:

import json
import pandas as pd
import os

from openai import OpenAI
from copy import deepcopy




class RecommendationAgent():
    def __init__(self,apriori_recommendation_path,popular_recommendation_path):
        self.client = OpenAI(

            api_key=os.getenv("GOOGLE_API_KEY"),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
        )
        self.model_name="gemini-1.5-flash"


        with open(apriori_recommendation_path, 'r') as file:
            self.apriori_recommendations = json.load(file)

        self.popular_recommendations = pd.read_csv(popular_recommendation_path)
        self.products = self.popular_recommendations['product'].tolist()
        # self.product_categories = self.popular_recommendations['product_category'].tolist()
        self.product_categories = list(set(self.popular_recommendations['product_category'].tolist()))




    def get_apriori_recommendation(self,products,top_k=5):
        recommendation_list = []
        for product in products:
            if product in self.apriori_recommendations:
                recommendation_list += self.apriori_recommendations[product]

        # Sort recommendation list by "confidence"
        recommendation_list = sorted(recommendation_list,key=lambda x: x['confidence'],reverse=True)

        recommendations = []
        recommendations_per_category = {}
        for recommendation in recommendation_list:
            # If Duplicated recommendations then skip
            if recommendation in recommendations:
                continue

            # Limit 2 recommendations per category
            product_catory = recommendation['product_category']
            if product_catory not in recommendations_per_category:
                recommendations_per_category[product_catory] = 0

            if recommendations_per_category[product_catory] >= 2:
                continue

            recommendations_per_category[product_catory]+=1

            # Add recommendation
            recommendations.append(recommendation['product'])

            if len(recommendations) >= top_k:
                break

        return recommendations

    def get_popular_recommendation(self,product_categories=None,top_k=5):
        recommendations_df = self.popular_recommendations

        if type(product_categories) == str:
            product_categories = [product_categories]

        if product_categories is not None:
            recommendations_df = self.popular_recommendations[self.popular_recommendations['product_category'].isin(product_categories)]
        recommendations_df = recommendations_df.sort_values(by='number_of_transactions',ascending=False)

        if recommendations_df.shape[0] == 0:
            return []

        recommendations = recommendations_df['product'].tolist()[:top_k]
        return recommendations

    def recommendation_classification(self,messages):
        system_prompt = """ You are a helpful AI assistant for a coffee shop application which serves drinks and pastries. We have 3 types of recommendations:

        1. Apriori Recommendations: These are recommendations based on the user's order history. We recommend items that are frequently bought together with the items in the user's order.
        2. Popular Recommendations: These are recommendations based on the popularity of items in the coffee shop. We recommend items that are popular among customers.
        3. Popular Recommendations by Category: Here the user asks to recommend them product in a category. Like what coffee do you recommend me to get?. We recommend items that are popular in the category of the user's requested category.

        Here is the list of items in the coffee shop:
        """+ ",".join(self.products) + """
        Here is the list of Categories we have in the coffee shop:
        """ + ",".join(self.product_categories) + """

        Your task is to determine which type of recommendation to provide based on the user's message.

        Your output should be in a structured json format like so. Each key is a string and each value is a string. Make sure to follow the format exactly:
        {
        "chain of thought": Write down your critical thinking about what type of recommendation is this input relevant to.
        "recommendation_type": "apriori" or "popular" or "popular by category". Pick one of those and only write the word.
        "parameters": This is a  python list. It's either a list of of items for apriori recommendations or a list of categories for popular by category recommendations. Leave it empty for popular recommendations. Make sure to use the exact strings from the list of items and categories above.
        }
        """

        input_messages = [{"role": "system", "content": system_prompt}] + messages[-3:]

        chatbot_output =get_chatbot_response(self.client,self.model_name,input_messages)
        # print("Chatbot raw output:", chatbot_output)#for testing purpose this one need
        chatbot_output =double_check_json_output(self.client,self.model_name,chatbot_output)
        # print("Chatbot output: ",chatbot_output)#for testing purpose this one need


        output = self.postprocess_classification(chatbot_output)
        # print("After postprocess classification: ",output)# for testing purpose this one need
        return output

    # def postprocess_classfication(self, output):
    #     try:
    #         output_dict = json.loads(output)
    #         print("Output from recommendation:", output_dict)

    #         # Check if 'recommendation_type' and 'parameters' are in the output
    #         recommendation_type = output_dict.get('recommendation_type', None)
    #         print("Recommendation type: ",recommendation_type)
    #         parameters = output_dict.get('parameters', [])
    #         print("Parameter: ",parameters)

    #         if recommendation_type is None:
    #             print("Warning: 'recommendation_type' is missing in the output.")

    #         dict_output = {
    #             "recommendation_type": recommendation_type,
    #             "parameters": parameters,
    #         }
    #         print("Printing dict output: ",dict_output)
    #         return dict_output
    #     except json.JSONDecodeError as e:
    #         print("JSON decoding error:", e)
    #         print("Received output:", output)
    #         return {"recommendation_type": "", "parameters": []}


    # def postprocess_classification(self, output):
    #     # Parse the outer JSON response
    #     parsed_output = json.loads(output)

    #     # Now extract and parse the 'message' field, which contains another JSON string
    #     message_content = json.loads(parsed_output.get('message', '{}'))

    #     # Extract 'recommendation_type' and 'parameters' from the message content
    #     dict_output = {
    #         "role": "assistant",
    #         "content": "",
    #         "recommendation": {
    #             "recommendation_type": message_content.get('recommendation_type', ''),
    #             "parameters": message_content.get('parameters', [])
    #         }
    #     }
    #     return dict_output
    # def postprocess_classification(self, output):
    #     # Check if output is empty or invalid
    #     if not output.strip():
    #         raise ValueError("Received empty or invalid output.")

    #     try:
    #         # Parse the outer JSON response
    #         parsed_output = json.loads(output)

    #         # Now extract and parse the 'message' field, which contains another JSON string
    #         message_content = json.loads(parsed_output.get('message', '{}'))

    #         # Extract 'recommendation_type' and 'parameters' from the message content
    #         dict_output = {
    #             "role": "assistant",
    #             "content": "",
    #             "recommendation": {
    #                 "recommendation_type": message_content.get('recommendation_type', ''),
    #                 "parameters": message_content.get('parameters', [])
    #             }
    #         }
    #         return dict_output

    #     except json.JSONDecodeError as e:
    #         print(f"JSON decode error: {str(e)}")
    #         return None
    #     except Exception as e:
    #         print(f"An error occurred: {str(e)}")
    #         return None

    # import json

    # def postprocess_classification(self, output):
    #     try:
    #         # Load the initial output JSON
    #         parsed_output = json.loads(output)

    #         # Extract and clean up the 'message' field which contains the nested JSON
    #         raw_message = parsed_output.get('message', '')

    #         # Check if the message is a valid JSON string, and remove unwanted characters (backticks)
    #         if raw_message.strip().startswith("```json"):
    #             clean_message = raw_message.strip("```json").strip()  # Clean the message to valid JSON
    #             clean_message = clean_message.strip("```").strip()  # Clean the message to valid JSON
    #         else:
    #             clean_message = raw_message  # Fallback if it's not in the expected format

    #         # Parse the cleaned message into a JSON object
    #         message_content = json.loads(clean_message)

    #         # Extract 'recommendation_type' and 'parameters' safely
    #         dict_output = {
    #             "role": "assistant",
    #             "content": "",  # Or populate as needed
    #             "recommendation": {
    #                 "recommendation_type": message_content.get('recommendation_type', ''),
    #                 "parameters": message_content.get('parameters', [])
    #             }
    #         }
    #         return dict_output

    #     except json.JSONDecodeError as e:
    #         print(f"JSON decode error: {str(e)}")
    #         return {"error": "Failed to parse recommendation content."}
    #     except Exception as e:
    #         print(f"An error occurred: {str(e)}")
    #         return {"error": "Unexpected error occurred."}

    def postprocess_classification(self, output):
        try:
            # Load the initial output JSON
            parsed_output = json.loads(output)

            # Extract and clean up the 'message' field which contains the nested JSON
            raw_message = parsed_output.get('message', '')

            # Check if the message is a valid JSON string, and remove unwanted characters (backticks)
            if raw_message.strip().startswith("```json"):
                clean_message = raw_message.strip("```json").strip()  # Clean the message to valid JSON
                clean_message = clean_message.strip("```").strip()  # Clean the message to valid JSON
                # Parse the cleaned message into a JSON object
                parsed_output = json.loads(clean_message)

            # Extract 'recommendation_type' and 'parameters' safely
            dict_output = {
                "role": "assistant",
                "content": "",  # Or populate as needed
                "recommendation": {
                    "recommendation_type": parsed_output.get('recommendation_type', ''),
                    "parameters": parsed_output.get('parameters', [])
                }
            }
            return dict_output

        except json.JSONDecodeError as e:
            print(f"JSON decode error: {str(e)}")
            return {"error": "Failed to parse recommendation content."}
        except Exception as e:
            print(f"An error occurred: {str(e)}")
            return {"error": "Unexpected error occurred."}

    # def postprocess_classification(self, output):
    #     try:
    #         # Strip any additional formatting like code block markers
    #         output = output.strip("```json").strip("```").strip()

    #         # Parse the JSON string into a dictionary
    #         output_dict = json.loads(output)
    #         print("Output from recommendation:", output_dict)

    #         # Extract 'recommendation_type' and 'parameters', ensuring defaults if missing
    #         recommendation_type = output_dict.get('recommendation_type', None)
    #         parameters = output_dict.get('parameters', [])

    #         print("Recommendation type:", recommendation_type)
    #         print("Parameters:", parameters)

    #         if recommendation_type is None:
    #             print("Warning: 'recommendation_type' is missing in the output.")

    #         # Prepare the dictionary for return
    #         dict_output = {
    #             "recommendation_type": recommendation_type,
    #             "parameters": parameters,
    #         }
    #         print("Printing dict output:", dict_output)
    #         return dict_output
    #     except json.JSONDecodeError as e:
    #         print("JSON decoding error:", e)
    #         print("Received output:", output)
    #         return {"recommendation_type": "", "parameters": []}
    #     except Exception as e:
    #         print("Unexpected error:", e)
    #         return {"recommendation_type": "", "parameters": []}


    def get_response(self,messages):
        messages = deepcopy(messages)

        recommendation_classification = self.recommendation_classification(messages)
        recommendation_type = recommendation_classification['recommendation']['recommendation_type']
        recommendations = []
        if recommendation_type == "apriori":
            recommendations = self.get_apriori_recommendation(recommendation_classification['recommendation']['parameters'])
        elif recommendation_type == "popular":
            recommendations = self.get_popular_recommendation()
        elif recommendation_type == "popular by category":
            recommendations = self.get_popular_recommendation(recommendation_classification['recommendation']['parameters'])

        if recommendations == []:
            return {"role": "assistant", "content":"Sorry, I can't help with that. Can I help you with your order?"}

        # Respond to User
        recommendations_str = ", ".join(recommendations)

        system_prompt = f"""
        You are a helpful AI assistant for a coffee shop application which serves drinks and pastries.
        your task is to recommend items to the user based on their input message. And respond in a friendly but concise way. And put it an unordered list with a very small description.

        I will provide what items you should recommend to the user based on their order in the user message.
        """

        prompt = f"""
        {messages[-1]['content']}

        Please recommend me those items exactly: {recommendations_str}
        """

        messages[-1]['content'] = prompt
        input_messages = [{"role": "system", "content": system_prompt}] + messages[-3:]

        chatbot_output =get_chatbot_response(self.client,self.model_name,input_messages)
        # print("get response output: ",chatbot_output)#for testing purpose this one need
        output = self.postprocess(chatbot_output)
        # print("get response After postprocess output: ",output)# for testing purpose this one need

        return output




    def get_recommendations_from_order(self,messages,order):
        messages = deepcopy(messages)
        products = []
        for product in order:
            products.append(product['item'])

        recommendations = self.get_apriori_recommendation(products)
        recommendations_str = ", ".join(recommendations)

        system_prompt = f"""
        You are a helpful AI assistant for a coffee shop application which serves drinks and pastries.
        your task is to recommend items to the user based on their order.

        I will provide what items you should recommend to the user based on their order in the user message.
        """

        prompt = f"""
        {messages[-1]['content']}

        Please recommend me those items exactly: {recommendations_str}
        """

        messages[-1]['content'] = prompt
        input_messages = [{"role": "system", "content": system_prompt}] + messages[-3:]

        chatbot_output =get_chatbot_response(self.client,self.model_name,input_messages)
        # print("recommendation from order get chatbot output: ",chatbot_output)#for testing purpose this one need
        output = self.postprocess(chatbot_output)
        # print("After postprocess from order: ",output)#for testing purpose this one need

        return output

    def postprocess(self, output):
        try:
            response = json.loads(output)
            message = response.get("message", "")
            return {
                "role": "assistant",
                "content": "Here are some recommendations based on your preferences:- "+message,
                "memory": {"agent": "recommendation_agent"}
            }
        except json.JSONDecodeError:
            return {
                "role": "assistant",
                "content": "I'm sorry, I couldn't process your request. Can I help you with your order?",
                "memory": {"agent": "recommendation_agent"}
            }



In [ ]:
# this work beter and best
from typing import Dict
if __name__ == "__main__":
    # Instantiate your agents
    guard_agent = GuardAgent()
    classification_agent = ClassificationAgent()
    # details_agent = DetailsAgent()

    # Map of agent names to their instances
    agent_dict: Dict[str, AgentProtocol] = {
        "details_agent": DetailsAgent(),
        "recommendation_agent": RecommendationAgent("/content/apriori_recommendations.json","/content/popularity_recommendation.csv")
    }

    messages = []

    while True:
        print("\n\nPrint messages.......")
        for message in messages:
            print(f"{message['role']}: {message['content']}")

        prompt = input("User: ")
        if prompt.lower() == "exit":
            print("Exiting the conversation.")
            break

        messages.append({"role": "user", "content": prompt})

        # Guard agent response
        guard_agent_response = guard_agent.get_response(messages)

        if guard_agent_response["memory"]["guard_decision"] == "not allowed":
            messages.append(guard_agent_response)
            continue

        # Classification agent response
        classification_agent_response = classification_agent.get_response(messages)
        chosen_agent = classification_agent_response["memory"]["classification_decision"]
        print("Chosen Agent: ", chosen_agent)

        # # Get the chosen agent response
        # agent = agent_dict[chosen_agent]
        # response = agent.get_response(messages)

        # messages.append(response)

        # Get the chosen agent response
        agent = agent_dict.get(chosen_agent)
        if agent is None:
            print(f"Error: Agent '{chosen_agent}' not found.")
            continue

        response = agent.get_response(messages)
        messages.append(response)



Print messages.......
User: what do u recommend me
User message: ```json
{
  "chain of thought": "The user is asking for a recommendation on what to buy. This falls under the purview of the recommendation agent.",
  "decision": "recommendation_agent",
  "message": ""
}
```
Decision made: recommendation_agent
Chosen Agent:  recommendation_agent
Chatbot raw output: {"message": "```json\n{\n  \"chain of thought\": \"The user's message is a general request for a recommendation without specifying any past orders or preferences. Therefore, a 'popular' recommendation is the most suitable type.\",\n  \"recommendation_type\": \"popular\",\n  \"parameters\": []\n}\n```\n"}
Chatbot output:  {"message": "The user's message is a general request for a recommendation without specifying any past orders or preferences. Therefore, a 'popular' recommendation is the most suitable type.", "recommendation_type": "popular", "parameters": []}

After postprocess classification:  {'role': 'assistant', 'conten

In [ ]:
# order taking agent
import os
import json

from openai import OpenAI
from copy import deepcopy



class OrderTakingAgent():
    def __init__(self, recommendation_agent):

        self.client = OpenAI(

            api_key=os.getenv("GOOGLE_API_KEY"),
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
        )
        self.model_name="gemini-1.5-flash"


        self.recommendation_agent = recommendation_agent

    def get_response(self,messages):
        messages = deepcopy(messages)

        system_prompt = """
            You are a customer support Bot for a coffee shop called "Merry's way"

            here is the menu for this coffee shop.

            Cappuccino - $4.50
            Jumbo Savory Scone - $3.25
            Latte - $4.75
            Chocolate Chip Biscotti - $2.50
            Espresso shot - $2.00
            Hazelnut Biscotti - $2.75
            Chocolate Croissant - $3.75
            Dark chocolate (Drinking Chocolate) - $5.00
            Cranberry Scone - $3.50
            Croissant - $3.25
            Almond Croissant - $4.00
            Ginger Biscotti - $2.50
            Oatmeal Scone - $3.25
            Ginger Scone - $3.50
            Chocolate syrup - $1.50
            Hazelnut syrup - $1.50
            Carmel syrup - $1.50
            Sugar Free Vanilla syrup - $1.50
            Dark chocolate (Packaged Chocolate) - $3.00

            Things to NOT DO:
            * DON't ask how to pay by cash or Card.
            * Don't tell the user to go to the counter
            * Don't tell the user to go to place to get the order


            You're task is as follows:
            1. Take the User's Order
            2. Validate that all their items are in the menu
            3. if an item is not in the menu let the user and repeat back the remaining valid order
            4. Ask them if they need anything else.
            5. If they do then repeat starting from step 3
            6. If they don't want anything else. Using the "order" object that is in the output. Make sure to hit all three points
                1. list down all the items and their prices
                2. calculate the total.
                3. Thank the user for the order and close the conversation with no more questions

            The user message will contain a section called memory. This section will contain the following:
            "order"
            "step number"
            please utilize this information to determine the next step in the process.

            produce the following output without any additions, not a single letter outside of the structure bellow.
            Your output should be in a structured json format like so. each key is a string and each value is a string. Make sure to follow the format exactly:
            {
            "chain of thought": Write down your critical thinking about what is the maximum task number the user is on write now. Then write down your critical thinking about the user input and it's relation to the coffee shop process. Then write down your thinking about how you should respond in the response parameter taking into consideration the Things to NOT DO section. and Focus on the things that you should not do.
            "step number": Determine which task you are on based on the conversation.
            "order": this is going to be a list of jsons like so. [{"item":put the item name, "quanitity": put the number that the user wants from this item, "price":put the total price of the item }]
            "response": write the a response to the user
            }
        """

        last_order_taking_status = ""
        asked_recommendation_before = False
        for message_index in range(len(messages)-1,0,-1):
            message = messages[message_index]

            agent_name = message.get("memory",{}).get("agent","")
            if message["role"] == "assistant" and agent_name == "order_taking_agent":
                step_number = message["memory"]["step number"]
                order = message["memory"]["order"]
                asked_recommendation_before = message["memory"]["asked_recommendation_before"]
                last_order_taking_status = f"""
                step number: {step_number}
                order: {order}
                """
                break

        messages[-1]['content'] = last_order_taking_status + " \n "+ messages[-1]['content']

        input_messages = [{"role": "system", "content": system_prompt}] + messages

        chatbot_output = get_chatbot_response(self.client,self.model_name,input_messages)

        # double check json
        chatbot_output = double_check_json_output(self.client,self.model_name,chatbot_output)

        output = self.postprocess(chatbot_output,messages,asked_recommendation_before)

        return output

    # def postprocess(self,output,messages,asked_recommendation_before):
    #     output = json.loads(output)

    #     if type(output["order"]) == str:
    #         output["order"] = json.loads(output["order"])

    #     response = output['response']
    #     if not asked_recommendation_before and len(output["order"])>0:
    #         recommendation_output = self.recommendation_agent.get_recommendations_from_order(messages,output['order'])
    #         response = recommendation_output['content']
    #         asked_recommendation_before = True

    #     dict_output = {
    #         "role": "assistant",
    #         "content": response ,
    #         "memory": {"agent":"order_taking_agent",
    #                    "step number": output["step number"],
    #                    "order": output["order"],
    #                    "asked_recommendation_before": asked_recommendation_before
    #                   }
    #     }


    #     return dict_output

    import json

    def postprocess(self, output, messages, asked_recommendation_before):
        try:
            # Parse the initial output JSON
            output_dict = json.loads(output)

            # Handle the 'order' field to ensure it's parsed correctly if it's a string
            if isinstance(output_dict.get("order"), str):
                output_dict["order"] = json.loads(output_dict["order"])

            # Get the response
            response = output_dict.get('response', '')

            # Handle recommendation logic if not previously asked and if order exists
            if not asked_recommendation_before and len(output_dict.get("order", [])) > 0:
                recommendation_output = self.recommendation_agent.get_recommendations_from_order(messages, output_dict["order"])
                response = recommendation_output.get('content', '')
                asked_recommendation_before = True

            # Construct the final dictionary output in a simplified structure
            dict_output = {
                "role": "assistant",
                "content": response,
                "memory": {
                    "agent": "order_taking_agent",
                    "step number": output_dict.get("step number", ""),
                    "order": output_dict.get("order", []),
                    "asked_recommendation_before": asked_recommendation_before
                }
            }

            return dict_output

        except json.JSONDecodeError as e:
            print("JSON decoding error:", e)
            print("Received output:", output)
            return {"error": "Failed to parse output."}

        except Exception as e:
            print(f"An error occurred: {e}")
            return {"error": "Unexpected error occurred."}








In [ ]:
# this work beter and best
from typing import Dict
if __name__ == "__main__":
    # Instantiate your agents
    guard_agent = GuardAgent()
    classification_agent = ClassificationAgent()
    recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json","/content/popularity_recommendation.csv")
    # details_agent = DetailsAgent()

    # Map of agent names to their instances
    agent_dict: Dict[str, AgentProtocol] = {
        "details_agent": DetailsAgent(),
        "recommendation_agent": recommendation_agent,
        "order_taking_agent": OrderTakingAgent(recommendation_agent)
    }

    messages = []

    while True:
        print("\n\nPrint messages.......")
        for message in messages:
            print(f"{message['role']}: {message['content']}")

        prompt = input("User: ")
        if prompt.lower() == "exit":
            print("Exiting the conversation.")
            break

        messages.append({"role": "user", "content": prompt})

        # Guard agent response
        guard_agent_response = guard_agent.get_response(messages)

        if guard_agent_response["memory"]["guard_decision"] == "not allowed":
            messages.append(guard_agent_response)
            continue

        # Classification agent response
        classification_agent_response = classification_agent.get_response(messages)
        chosen_agent = classification_agent_response["memory"]["classification_decision"]
        # print("Chosen Agent: ", chosen_agent)#for testing purposr this one needed

        # # Get the chosen agent response
        # agent = agent_dict[chosen_agent]
        # response = agent.get_response(messages)

        # messages.append(response)

        # Get the chosen agent response
        agent = agent_dict.get(chosen_agent)
        if agent is None:
            print(f"Error: Agent '{chosen_agent}' not found.")
            continue

        response = agent.get_response(messages)
        messages.append(response)



Print messages.......
User: what's 2*6


Print messages.......
user: what's 2*6
assistant: I'm sorry, I cannot answer that question.  Welcome to Merry's Way! What can I get for you today?
User: what do u recommend me


Print messages.......
user: what's 2*6
assistant: I'm sorry, I cannot answer that question.  Welcome to Merry's Way! What can I get for you today?
user: what do u recommend me
assistant: Here are some recommendations based on your preferences:- * **Cappuccino:** A classic blend of espresso, steamed milk, and foamed milk.
* **Latte:** Espresso with steamed milk and a thin layer of foam.
* **Dark Chocolate:** Rich and intense dark chocolate.
* **Chocolate Croissant:** Flaky pastry filled with decadent chocolate.
* **Espresso Shot:** A concentrated shot of espresso.

User: whats the price latte


Print messages.......
user: what's 2*6
assistant: I'm sorry, I cannot answer that question.  Welcome to Merry's Way! What can I get for you today?
user: what do u recommend me
as

In [ ]:
# this work beter and best
from IPython.display import clear_output
from typing import Dict
if __name__ == "__main__":
    # Instantiate your agents
    guard_agent = GuardAgent()
    classification_agent = ClassificationAgent()
    recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json","/content/popularity_recommendation.csv")
    # details_agent = DetailsAgent()

    # Map of agent names to their instances
    agent_dict: Dict[str, AgentProtocol] = {
        "details_agent": DetailsAgent(),
        "recommendation_agent": recommendation_agent,
        "order_taking_agent": OrderTakingAgent(recommendation_agent)
    }

    messages = []

    while True:
        # from IPython.display import clear_output
        # clear_output(wait=True)
        print("\n\nPrint messages.......")
        for message in messages:
            print(f"{message['role']}: {message['content']}")

        prompt = input("User: ")
        if prompt.lower() == "exit":
            print("Exiting the conversation.")
            break

        messages.append({"role": "user", "content": prompt})

        # Guard agent response
        guard_agent_response = guard_agent.get_response(messages)

        if guard_agent_response["memory"]["guard_decision"] == "not allowed":
            messages.append(guard_agent_response)
            continue

        # Classification agent response
        classification_agent_response = classification_agent.get_response(messages)
        chosen_agent = classification_agent_response["memory"]["classification_decision"]


        # Get the chosen agent response
        agent = agent_dict.get(chosen_agent)
        if agent is None:
            print(f"Error: Agent '{chosen_agent}' not found.")
            continue

        response = agent.get_response(messages)
        messages.append(response)




Print messages.......
User: what's 1+4


Print messages.......
user: what's 1+4
assistant: I'm sorry, I cannot answer that question.  Welcome to Merry's Way! What can I get for you today?
User: what do u recommend me


Print messages.......
user: what's 1+4
assistant: I'm sorry, I cannot answer that question.  Welcome to Merry's Way! What can I get for you today?
user: what do u recommend me
assistant: Here are some recommendations based on your preferences:- I recommend:

* **Cappuccino:** A classic combination of espresso, steamed milk, and foamed milk.
* **Latte:** Espresso with steamed milk and a thin layer of foam.
* **Dark Chocolate:** Rich and intense dark chocolate.
* **Chocolate Croissant:** Flaky pastry filled with delicious chocolate.
* **Espresso Shot:** A concentrated shot of espresso.

User: what type of coffe do u suggested me


Print messages.......
user: what's 1+4
assistant: I'm sorry, I cannot answer that question.  Welcome to Merry's Way! What can I get for you to

In [ ]:
# this work beter and best
from IPython.display import clear_output
from typing import Dict
if __name__ == "__main__":
    # Instantiate your agents
    guard_agent = GuardAgent()
    classification_agent = ClassificationAgent()
    recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json","/content/popularity_recommendation.csv")
    # details_agent = DetailsAgent()

    # Map of agent names to their instances
    agent_dict: Dict[str, AgentProtocol] = {
        "details_agent": DetailsAgent(),
        "recommendation_agent": recommendation_agent,
        "order_taking_agent": OrderTakingAgent(recommendation_agent)
    }

    messages = []

    while True:
        # from IPython.display import clear_output
        # clear_output(wait=True)
        print("\n\nPrint messages.......")
        for message in messages:
            print(f"{message['role']}: {message['content']}")

        prompt = input("User: ")
        if prompt.lower() == "exit":
            print("Exiting the conversation.")
            break

        messages.append({"role": "user", "content": prompt})

        # Guard agent response
        guard_agent_response = guard_agent.get_response(messages)

        if guard_agent_response["memory"]["guard_decision"] == "not allowed":
            messages.append(guard_agent_response)
            continue

        # Classification agent response
        classification_agent_response = classification_agent.get_response(messages)
        chosen_agent = classification_agent_response["memory"]["classification_decision"]


        # Get the chosen agent response
        agent = agent_dict.get(chosen_agent)
        if agent is None:
            print(f"Error: Agent '{chosen_agent}' not found.")
            continue

        response = agent.get_response(messages)
        messages.append(response)



Print messages.......
User: exit
Exiting the conversation.


In [ ]:
from IPython.display import clear_output
from typing import Dict, List

def main():
    # Instantiate your agents
    guard_agent = GuardAgent()
    classification_agent = ClassificationAgent()
    recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json","/content/popularity_recommendation.csv")
    # details_agent = DetailsAgent()

    # Map of agent names to their instances
    agent_dict: Dict[str, AgentProtocol] = {
        "details_agent": DetailsAgent(),
        "recommendation_agent": recommendation_agent,
        "order_taking_agent": OrderTakingAgent(recommendation_agent)
    }

    messages = []

    while True:
        # from IPython.display import clear_output
        clear_output(wait=True)
        print("\n\nPrint messages.......")
        for message in messages:
            print(f"{message['role']}: {message['content']}")

        prompt = input("User: ")
        if prompt.lower() == "exit":
            print("Exiting the conversation.")
            break

        messages.append({"role": "user", "content": prompt})

        # Guard agent response
        guard_agent_response = guard_agent.get_response(messages)

        if guard_agent_response["memory"]["guard_decision"] == "not allowed":
            messages.append(guard_agent_response)
            continue

        # Classification agent response
        classification_agent_response = classification_agent.get_response(messages)
        chosen_agent = classification_agent_response["memory"]["classification_decision"]


        # Get the chosen agent response
        agent = agent_dict.get(chosen_agent)
        if agent is None:
            print(f"Error: Agent '{chosen_agent}' not found.")
            continue

        response = agent.get_response(messages)
        messages.append(response)

if __name__ == "__main__":
    main()






Print messages.......
user: whats the price 1+5
assistant: {"message": "Hello there!  One cappuccino is $4.50.  I'm not sure what \"1+5\" refers to in terms of your order, could you clarify?  Perhaps you'd like 6 cappuccinos?  That would be $27.00.  Let me know how I can help further!\n"}


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



In [ ]:
!pip install gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


In [ ]:
import gradio as gr
from typing import List, Dict

# Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

messages = []

def chat_interface(user_input: str) -> str:
    global messages

    if user_input.lower() == "exit":
        return "Exiting the conversation."

    messages.append({"role": "user", "content": user_input})

    # Guard agent response
    guard_agent_response = guard_agent.get_response(messages)

    if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        messages.append(guard_agent_response)
        return guard_agent_response["content"]

    # Classification agent response
    classification_agent_response = classification_agent.get_response(messages)
    chosen_agent = classification_agent_response["memory"]["classification_decision"]

    # Get the chosen agent response
    agent = agent_dict.get(chosen_agent)
    if agent is None:
        return f"Error: Agent '{chosen_agent}' not found."

    response = agent.get_response(messages)
    messages.append(response)
    return response["content"]

# Gradio interface
gr_interface = gr.Interface(
    fn=chat_interface,
    inputs="text",
    outputs="text",
    title="Conversational Agents",
    description="Chat with multiple agents handling different tasks."
)

# Launch the app
gr_interface.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://584cdf7230e007d9f8.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import random
import time
from typing import List, Dict

# Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

# Simulating the agents and message history
messages = []

def respond(message, chat_history):
    global messages

    # Check for exit command
    if message.lower() == "exit":
        chat_history.append({"role": "assistant", "content": "Exiting the conversation."})
        return "", chat_history

    chat_history.append({"role": "user", "content": message})

    # Guard agent response
    guard_agent_response = guard_agent.get_response(chat_history)
    if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        chat_history.append({"role": "assistant", "content": guard_agent_response["content"]})
        return guard_agent_response["content"], chat_history

    # Classification agent response
    classification_agent_response = classification_agent.get_response(chat_history)
    chosen_agent = classification_agent_response["memory"]["classification_decision"]

    # Get the chosen agent response
    agent = agent_dict.get(chosen_agent)
    if agent is None:
        error_msg = f"Error: Agent '{chosen_agent}' not found."
        chat_history.append({"role": "assistant", "content": error_msg})
        return error_msg, chat_history

    response = agent.get_response(chat_history)
    chat_history.append({"role": "assistant", "content": response["content"]})

    # Return the updated chat history
    time.sleep(2)
    return "", chat_history

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fbc80e3b236ef68bac.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import random
import time
from typing import List, Dict

# Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

# Simulating the agents and message history
messages = []

def respond(message, chat_history):
    global messages

    # Check for exit command
    if message.lower() == "exit":
        chat_history.append({"role": "assistant", "content": "Exiting the conversation."})
        return "", chat_history

    chat_history.append({"role": "user", "content": message})

    # Guard agent response
    guard_agent_response = guard_agent.get_response(chat_history)
    if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        chat_history.append({"role": "assistant", "content": guard_agent_response["content"]})
        return guard_agent_response["content"], chat_history

    # Classification agent response
    classification_agent_response = classification_agent.get_response(chat_history)
    chosen_agent = classification_agent_response["memory"]["classification_decision"]

    # Get the chosen agent response
    agent = agent_dict.get(chosen_agent)
    if agent is None:
        error_msg = f"Error: Agent '{chosen_agent}' not found."
        chat_history.append({"role": "assistant", "content": error_msg})
        return error_msg, chat_history

    response = agent.get_response(chat_history)
    chat_history.append({"role": "assistant", "content": response["content"]})

    # Markdown formatting (wrap response content in markdown)
    markdown_response = f"### Assistant's Response\n{response['content']}"

    # Return the updated chat history with markdown response
    time.sleep(2)
    return markdown_response, chat_history

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a0c1c83edd1f9bea3d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# import gradio as gr
# import random
# import time
# from typing import List, Dict

# # Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
# guard_agent = GuardAgent()
# classification_agent = ClassificationAgent()
# recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
# details_agent = DetailsAgent()
# order_taking_agent = OrderTakingAgent(recommendation_agent)

# # Map of agent names to their instances
# agent_dict: Dict[str, AgentProtocol] = {
#     "details_agent": details_agent,
#     "recommendation_agent": recommendation_agent,
#     "order_taking_agent": order_taking_agent
# }

# # Simulating the agents and message history
# messages = []

# def respond(message, chat_history):
#     global messages

#     # Check for exit command
#     if message.lower() == "exit":
#         chat_history.append({"role": "assistant", "content": "Exiting the conversation."})
#         return "", chat_history

#     chat_history.append({"role": "user", "content": message})

#     # Guard agent response
#     guard_agent_response = guard_agent.get_response(chat_history)
#     if guard_agent_response["memory"]["guard_decision"] == "not allowed":
#         chat_history.append({"role": "assistant", "content": guard_agent_response["content"]})
#         return guard_agent_response["content"], chat_history

#     # Classification agent response
#     classification_agent_response = classification_agent.get_response(chat_history)
#     chosen_agent = classification_agent_response["memory"]["classification_decision"]

#     # Get the chosen agent response
#     agent = agent_dict.get(chosen_agent)
#     if agent is None:
#         error_msg = f"Error: Agent '{chosen_agent}' not found."
#         chat_history.append({"role": "assistant", "content": error_msg})
#         return error_msg, chat_history

#     response = agent.get_response(chat_history)
#     chat_history.append({"role": "assistant", "content": response["content"]})

#     # Markdown formatting (wrap response content in markdown)
#     markdown_response = f"### Assistant's Response\n{response['content']}"

#     # # Return the updated chat history with markdown response
#     # time.sleep(2)
#     # return "", chat_history  # Clear the textbox but keep chat history updated

#     time.sleep(2)
#     return markdown_response, chat_history

# with gr.Blocks() as demo:
#     chatbot = gr.Chatbot(type="messages")
#     msg = gr.Textbox()
#     clear = gr.ClearButton([msg, chatbot])

#     # Clear the message after the user submits it
#     msg.submit(respond, [msg, chatbot], [msg, chatbot])

# demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://690515d651d4fe6770.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# this work good and better
import gradio as gr
import random
import time
from typing import List, Dict

# Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

# Simulating the agents and message history
messages = []

def respond(message, chat_history):
    global messages

    # Check for exit command
    if message.lower() == "exit":
        chat_history.append({"role": "assistant", "content": "Exiting the conversation."})
        return "", chat_history

    chat_history.append({"role": "user", "content": message})

    # Guard agent response
    guard_agent_response = guard_agent.get_response(chat_history)
    if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        chat_history.append({"role": "assistant", "content": guard_agent_response["content"]})
        return guard_agent_response["content"], chat_history

    # Classification agent response
    classification_agent_response = classification_agent.get_response(chat_history)
    chosen_agent = classification_agent_response["memory"]["classification_decision"]

    # Get the chosen agent response
    agent = agent_dict.get(chosen_agent)
    if agent is None:
        error_msg = f"Error: Agent '{chosen_agent}' not found."
        chat_history.append({"role": "assistant", "content": error_msg})
        return error_msg, chat_history

    response = agent.get_response(chat_history)
    chat_history.append({"role": "assistant", "content": response["content"]})

    # Markdown formatting (wrap response content in markdown)
    markdown_response = f"### Assistant's Response\n{response['content']}"

    # Return the updated chat history with markdown response
    time.sleep(2)
    return "", chat_history  # Clear the textbox but keep chat history updated

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    # Clear the message after the user submits it
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f59e591971cf1f05c0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import random
import time
import json
from typing import List, Dict

# Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

# Simulating the agents and message history
messages = []

def respond(message, chat_history):
    global messages

    # Check for exit command
    if message.lower() == "exit":
        chat_history.append(("user", message))
        chat_history.append(("assistant", "Exiting the conversation."))
        return "", chat_history

    chat_history.append(("user", message))

    # Guard agent response
    guard_agent_response = guard_agent.get_response(chat_history)
    if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        chat_history.append(("assistant", guard_agent_response["content"]))
        return "", chat_history

    # Classification agent response
    classification_agent_response = classification_agent.get_response(chat_history)
    chosen_agent = classification_agent_response["memory"]["classification_decision"]

    # Get the chosen agent response
    agent = agent_dict.get(chosen_agent)
    if agent is None:
        error_msg = f"Error: Agent '{chosen_agent}' not found."
        chat_history.append(("assistant", error_msg))
        return "", chat_history

    response = agent.get_response(chat_history)

    # Process the response content
    try:
        # Try to parse JSON response
        content_dict = json.loads(response["content"])
        processed_content = content_dict.get("message", response["content"])
    except (json.JSONDecodeError, AttributeError):
        # Fallback if not JSON
        processed_content = response["content"]

    # Convert to markdown formatting
    markdown_content = processed_content.replace("\\n", "\n")  # Unescape newlines
    markdown_content = "\n".join([f"- {line.strip()}" for line in markdown_content.split("\n") if line.strip()])

    chat_history.append(("assistant", markdown_content))
    time.sleep(1)
    return "", chat_history

with gr.Blocks() as demo:
    chatbot = gr.Chatbot()
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    msg.submit(respond, [msg, chatbot], [msg, chatbot], queue=False)

demo.launch()

<ipython-input-39-7706bb0cc6fd>:72: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7c0f0ba027dca373d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# @ this is the final one and better and best
# Best production ready finale gradio app

import gradio as gr
import time
from typing import List, Dict

# Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

# Simulating the agents and message history
messages = []

# Define item prices (correcting syntax error)
item_prices = {
    "Cappuccino": 4.50,
    "Jumbo Savory Scone": 3.25,
    "Latte": 4.75,
    "Chocolate Chip Biscotti": 2.50,
    "Espresso shot": 2.00,
    "Hazelnut Biscotti": 2.75,
    "Chocolate Croissant": 3.75,
    "Dark chocolate (Drinking Chocolate)": 5.00,
    "Cranberry Scone": 3.50,
    "Croissant": 3.25,
    "Almond Croissant": 4.00,
    "Ginger Biscotti": 2.50,
    "Oatmeal Scone": 3.25,
    "Ginger Scone": 3.50,
    "Chocolate syrup": 1.50,
    "Hazelnut syrup": 1.50,
    "Caramel syrup": 1.50,
    "Sugar Free Vanilla syrup": 1.50,
    "Dark chocolate (Packaged Chocolate)": 3.00
}

def respond(message, chat_history):
    global messages

    # Check for exit command
    if message.lower() == "exit":
        chat_history.append({"role": "assistant", "content": "Exiting the conversation."})
        return "", chat_history

    chat_history.append({"role": "user", "content": message})

    # Guard agent response
    guard_agent_response = guard_agent.get_response(chat_history)
    if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        chat_history.append({"role": "assistant", "content": guard_agent_response["content"]})
        return guard_agent_response["content"], chat_history

    # Classification agent response
    classification_agent_response = classification_agent.get_response(chat_history)
    chosen_agent = classification_agent_response["memory"]["classification_decision"]

    # Get the chosen agent response
    agent = agent_dict.get(chosen_agent)
    if agent is None:
        error_msg = f"Error: Agent '{chosen_agent}' not found."
        chat_history.append({"role": "assistant", "content": error_msg})
        return error_msg, chat_history

    response = agent.get_response(chat_history)
    chat_history.append({"role": "assistant", "content": response["content"]})

    # Initialize a list to store order summaries
    order_summary = []

    # Process the items mentioned in the message
    for item, price in item_prices.items():
        if item.lower() in message.lower():
            # Extract quantities, assume format "1+5" or any other numbers
            items_and_qty = [int(num) for num in message.split() if num.isdigit()]
            total_qty = sum(items_and_qty) if items_and_qty else 1  # Default to 1 if no quantity is found
            total_price = total_qty * price

            # Add item to order summary in markdown
            order_summary.append(f"One {item} is ${price:.2f}. You ordered {total_qty} {item}s. The total price is ${total_price:.2f}.")

    # Combine the order summary with the assistant's response
    markdown_response = "\n\n".join(order_summary) if order_summary else f"### Assistant's Response\n{response['content']}"

    # Return the updated chat history with markdown response
    time.sleep(2)
    # return markdown_response, chat_history  # Return markdown response with chat history updated
    return "", chat_history

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    # Clear the message after the user submits it
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6697493270202601eb.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# this is good

import gradio as gr
import time
from typing import List, Dict

# Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

# Simulating the agents and message history
messages = []

# Define item prices (correcting syntax error)
item_prices = {
    "Cappuccino": 4.50,
    "Jumbo Savory Scone": 3.25,
    "Latte": 4.75,
    "Chocolate Chip Biscotti": 2.50,
    "Espresso shot": 2.00,
    "Hazelnut Biscotti": 2.75,
    "Chocolate Croissant": 3.75,
    "Dark chocolate (Drinking Chocolate)": 5.00,
    "Cranberry Scone": 3.50,
    "Croissant": 3.25,
    "Almond Croissant": 4.00,
    "Ginger Biscotti": 2.50,
    "Oatmeal Scone": 3.25,
    "Ginger Scone": 3.50,
    "Chocolate syrup": 1.50,
    "Hazelnut syrup": 1.50,
    "Caramel syrup": 1.50,
    "Sugar Free Vanilla syrup": 1.50,
    "Dark chocolate (Packaged Chocolate)": 3.00
}

def respond(message, chat_history):
    global messages

    # Check for exit command
    if message.lower() == "exit":
        chat_history.append({"role": "assistant", "content": "Exiting the conversation."})
        return "", chat_history

    chat_history.append({"role": "user", "content": message})

    # Guard agent response
    guard_agent_response = guard_agent.get_response(chat_history)
    if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        chat_history.append({"role": "assistant", "content": guard_agent_response["content"]})
        return guard_agent_response["content"], chat_history

    # Classification agent response
    classification_agent_response = classification_agent.get_response(chat_history)
    chosen_agent = classification_agent_response["memory"]["classification_decision"]

    # Get the chosen agent response
    agent = agent_dict.get(chosen_agent)
    if agent is None:
        error_msg = f"Error: Agent '{chosen_agent}' not found."
        chat_history.append({"role": "assistant", "content": error_msg})
        return error_msg, chat_history

    response = agent.get_response(chat_history)
    chat_history.append({"role": "assistant", "content": response["content"]})

    # Initialize a list to store order summaries
    order_summary = []

    # Process the items mentioned in the message
    for item, price in item_prices.items():
        if item.lower() in message.lower():
            # Extract quantities, assume format "1+5" or any other numbers
            items_and_qty = [int(num) for num in message.split() if num.isdigit()]
            total_qty = sum(items_and_qty) if items_and_qty else 1  # Default to 1 if no quantity is found
            total_price = total_qty * price

            # Add item to order summary in markdown
            order_summary.append(f"One {item} is ${price:.2f}. You ordered {total_qty} {item}s. The total price is ${total_price:.2f}.")

    # Only return the order summary or markdown response, not the assistant's generic message
    markdown_response = "\n\n".join(order_summary) if order_summary else ""

    # If no items were ordered, include the assistant's response (this ensures an appropriate response is shown)
    if not markdown_response and response["content"].strip() != "":
        markdown_response = f"### Assistant's Response\n{response['content']}"

    # Add the generated markdown response to the chat history
    chat_history.append({"role": "assistant", "content": markdown_response})

    # Return the updated chat history with markdown response
    time.sleep(2)
    return "", chat_history  # Clear the textbox but keep chat history updated

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    # Clear the message after the user submits it
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://49139a0e4e74c06787.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import random
import time

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    def respond(message, chat_history):
        bot_message = random.choice(["How are you?", "Today is a great day", "I'm very hungry"])
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_message})
        time.sleep(2)
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d46ab40d2ecf95e26b.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
from typing import List, Dict

# Initialize agents
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

agent_dict: Dict[str, 'AgentProtocol'] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

messages = []

def chat_interface(user_input: str) -> str:
    global messages

    if user_input.lower() == "exit":
        return "Exiting the conversation."

    # Log user input
    messages.append({"role": "user", "content": user_input})

    # Guard agent's response
    guard_response = guard_agent.get_response(messages)
    messages.append({"role": "guard_agent", "content": guard_response["content"]})

    if guard_response["memory"]["guard_decision"] == "not allowed":
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in messages])

    # Classification agent's response
    classification_response = classification_agent.get_response(messages)
    messages.append({"role": "classification_agent", "content": classification_response["content"]})

    chosen_agent_name = classification_response["memory"]["classification_decision"]
    chosen_agent = agent_dict.get(chosen_agent_name)

    if not chosen_agent:
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in messages])

    # Get the chosen agent's response
    agent_response = chosen_agent.get_response(messages)
    messages.append({"role": chosen_agent_name, "content": agent_response["content"]})

    # Return the entire conversation as a string
    return "\n".join([f"{msg['role']}: {msg['content']}" for msg in messages])

# Gradio interface with conversation history
gr_interface = gr.Interface(
    fn=chat_interface,
    inputs=gr.Textbox(placeholder="Type your message here...", lines=1, interactive=True),
    outputs=gr.Textbox(label="Conversation", lines=15, interactive=False),
    title="Conversational Chatbot",
    description="Chat with multiple agents handling different tasks in a conversational interface."
)

# Launch the app
gr_interface.launch(debug=True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3a24aa62270b9a55ea.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 2042, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 1589, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/anyio/to_thread.py", line 33, in run_sync
    return await get_asynclib().run_sync_in_worker_thread(
           ^^^^^^^^^^

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://562a57a20bdc409775.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://850f105d2f72b2eb50.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://bc998618d3baaf6e9e.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://3a24aa62270b9a55ea.gradio.live


In [ ]:
import gradio as gr
import random
import time

from IPython.display import clear_output
from typing import Dict, List

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()
    clear = gr.ClearButton([msg, chatbot])

    # Instantiate your agents
    guard_agent = GuardAgent()
    classification_agent = ClassificationAgent()
    recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json","/content/popularity_recommendation.csv")
    # details_agent = DetailsAgent()

    # Map of agent names to their instances
    agent_dict: Dict[str, AgentProtocol] = {
        "details_agent": DetailsAgent(),
        "recommendation_agent": recommendation_agent,
        "order_taking_agent": OrderTakingAgent(recommendation_agent)
    }

    def respond(message, chat_history):

        chat_history.append({"role": "user", "content": message})

        # if prompt.lower() == "exit":
        #     print("Exiting the conversation.")
        #     break
        try:
            # Guard agent response
            guard_agent_response = guard_agent.get_response(messages)

            if guard_agent_response["memory"]["guard_decision"] == "not allowed":
                chat_history.append({"role": "assistant", "content": guard_agent_response})
                return "", chat_history

            # Classification agent response
            classification_agent_response = classification_agent.get_response(messages)
            chosen_agent = classification_agent_response["memory"]["classification_decision"]


            # Get the chosen agent response
            agent = agent_dict.get(chosen_agent)
            if agent is None:
                print(f"Error: Agent '{chosen_agent}' not found.")
                chat_history.append({"role": "assistant", "content": f"Error: Agent '{chosen_agent}' not found."})
                return "", chat_history

            response = agent.get_response(messages)

            chat_history.append({"role": "assistant", "content": response})
            time.sleep(2)
            return "", chat_history
        except Exception as e:
            chat_history.append({"role": "assistant", "content": str(e)})
            return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://375a05fc6dfa40b3e7.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
from typing import List, Generator, Tuple, Dict

# Initialize agents (assuming you've defined GuardAgent, ClassificationAgent, etc.)
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")
details_agent = DetailsAgent()
order_taking_agent = OrderTakingAgent(recommendation_agent)

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": details_agent,
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": order_taking_agent
}

messages = []

def stream_response(user_input: str, history: List[Tuple[str, str]]) -> Generator[str, None, None]:
    global messages

    if user_input.lower() == "exit":
        yield "Exiting the conversation."
        return

    messages.append({"role": "user", "content": user_input})
    yield f"User: {user_input}"

    # Guard agent response
    guard_agent_response = guard_agent.get_response(messages)

    if guard_agent_response["memory"]["guard_decision"] == "not allowed":
        messages.append(guard_agent_response)
        yield f"Bot: {guard_agent_response['content']}"
        return

    # Classification agent response
    classification_agent_response = classification_agent.get_response(messages)
    chosen_agent = classification_agent_response["memory"]["classification_decision"]

    # Get the chosen agent response
    agent = agent_dict.get(chosen_agent)
    if agent is None:
        yield f"Error: Agent '{chosen_agent}' not found."
        return

    response = agent.get_response(messages)
    messages.append(response)

    # Stream each part of the response content
    for partial in response["content"]:
        yield f"Bot: {partial}"

# Gradio interface
demo_interface = gr.ChatInterface(
    stream_response,
    textbox=gr.Textbox(placeholder="Send a message...", container=False, autoscroll=True, scale=7)
)

# Launch the app
demo_interface.launch(share=True, debug=True)


/usr/local/lib/python3.10/dist-packages/gradio/components/chatbot.py:279: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://867a2eff95613cad2a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://867a2eff95613cad2a.gradio.live


In [ ]:
!pip -q install gradio langchain-openai langchain-community langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.2 MB/s eta 0:00:00


In [ ]:
import gradio as gr
from typing import Dict
# from langchain.agents import AgentProtocol
# from langchain.agents import GuardAgent, ClassificationAgent, RecommendationAgent, OrderTakingAgent
from langchain.schema import HumanMessage, AIMessage, SystemMessage  # Corrected import

# Initialize your agents
guard_agent = GuardAgent()
classification_agent = ClassificationAgent()
recommendation_agent = RecommendationAgent("/content/apriori_recommendations.json", "/content/popularity_recommendation.csv")

# Map of agent names to their instances
agent_dict: Dict[str, AgentProtocol] = {
    "details_agent": DetailsAgent(),
    "recommendation_agent": recommendation_agent,
    "order_taking_agent": OrderTakingAgent(recommendation_agent)
}

messages = []

# Inside the stream_response function
def stream_response(message, history):
    print(f"Input: {message}. History: {history}\n")

    history_langchain_format = []

    # Append system message to format the conversation history
    history_langchain_format.append(SystemMessage(content="Your system message here"))

    # Append human and AI messages from the history
    for human, ai in history:
        history_langchain_format.append(HumanMessage(content=human))

        # Ensure that AI message content is not None or invalid
        if ai is not None and isinstance(ai, str):
            history_langchain_format.append(AIMessage(content=ai))
        else:
            print(f"Warning: Invalid AI message content: {ai}")

    # Append the new message from the user
    if message is not None:
        history_langchain_format.append(HumanMessage(content=message))

    partial_message = ""

    # Simulate agent decision-making and response streaming
    while True:
        # Handle agent decision-making based on user input
        guard_agent_response = guard_agent.get_response(messages)

        if guard_agent_response["memory"]["guard_decision"] == "not allowed":
            messages.append(guard_agent_response)
            continue

        # Classification agent response to determine which agent to call
        classification_agent_response = classification_agent.get_response(messages)
        chosen_agent = classification_agent_response["memory"]["classification_decision"]

        # Get the chosen agent's response
        agent = agent_dict.get(chosen_agent)
        if agent is None:
            print(f"Error: Agent '{chosen_agent}' not found.")
            continue

        response = agent.get_response(messages)

        # Ensure that the response is valid before appending
        if response and "content" in response and isinstance(response["content"], str):
            messages.append(response)
        else:
            print(f"Error: Invalid response from agent: {response}")
            continue

        # Simulate streaming the agent's response
        for response_part in response['content']:
            partial_message += response_part
            yield partial_message
# Gradio interface setup
demo_interface = gr.ChatInterface(
    stream_response,
    textbox=gr.Textbox(placeholder="Message the LLM...", container=False, scale=7),
)

demo_interface.launch(share=True, debug=True)


/usr/local/lib/python3.10/dist-packages/gradio/components/chatbot.py:279: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6826af242c0f4f11f8.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Input: whats the price latte. History: []



Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
  File "/usr/local/lib/python3.10/dist-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
  File "/usr/local/lib/python3.10/dist-packages/gradio/blocks.py", line 2042, in process_api
    result = await self.call_function(
  File "/usr/local/lib/python3.10/dist-packages/gradio/blocks.py", line 1601, in call_function
    prediction = await utils.async_iteration(iterator)
  File "/usr/local/lib/python3.10/dist-packages/gradio/utils.py", line 728, in async_iteration
    return await anext(iterator)
  File "/usr/local/lib/python3.10/dist-packages/gradio/utils.py", line 833, in asyncgen_wrapper
    response = await iterator.__anext__()
  File "/usr/local/lib/python3.10/dist-packages/gradio/chat_interface.py", line 884, in _stream_fn
    first_response 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7867 <> https://6826af242c0f4f11f8.gradio.live


NameError: name 'GuardAgent' is not defined